In [0]:
SELECT '================================================' AS message;
SELECT 'Loading into SILVER Layer' AS message;
SELECT '================================================' AS message;

SELECT '>> Truncating Table: Silver.orders' AS message;

TRUNCATE TABLE silver.orders;

SELECT '>> Inserting Data Into: Silver.orders' AS message;

INSERT INTO silver.orders
SELECT 
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    
    -- RULE: Fix Approved At (Ensure it's actually after Purchase)
    CASE 
      WHEN order_approved_at > order_purchase_timestamp THEN order_approved_at
      ELSE NULL -- Or set to order_purchase_timestamp depending on business preference
    END AS order_approved_at,

    -- RULE: Fix Carrier Date (Logic check against status and previous step)
    CASE 
      WHEN order_delivered_carrier_date > order_approved_at
       AND order_status NOT IN ('unavailable','canceled','created')
      THEN order_delivered_carrier_date
      ELSE NULL 
    END AS order_delivered_carrier_date,

    CASE 
        WHEN order_delivered_customer_date < order_delivered_carrier_date 
        THEN order_delivered_carrier_date 
        ELSE order_delivered_customer_date 
    END AS order_delivered_customer_date,
    order_estimated_delivery_date
FROM bronze.orders
WHERE 
    -- RULE 1: Remove orders marked delivered but missing the timestamp
    NOT (order_status = 'delivered' AND order_delivered_customer_date IS NULL)

    -- RULE 2: Remove rows with a date that are NOT marked 'delivered'
    AND NOT (order_delivered_customer_date IS NOT NULL AND order_status != 'delivered')
    
    -- RULE 3 (DATA INTEGRITY): Ensure the order_id is valid
    AND order_id IS NOT NULL;

SELECT '>> Truncating Table: Silver.customers' AS message;

TRUNCATE TABLE silver.customers;

SELECT '>> Inserting Data Into: Silver.customers' AS message;
INSERT INTO silver.customers
SELECT 
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    -- Standardize text to UPPERCASE for cleaner grouping
    LOWER(TRIM(customer_city)) AS customer_city,
    UPPER(TRIM(customer_state)) AS customer_state
FROM bronze.customers
WHERE 
    customer_id IS NOT NULL 
    AND customer_unique_id IS NOT NULL
-- Ensure no duplicate customer_id records enter Silver
QUALIFY ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY customer_zip_code_prefix DESC) = 1;


SELECT '>> Truncating Table: Silver.geolocation' AS message;

TRUNCATE TABLE silver.geolocation;

SELECT '>> Inserting Data Into: Silver.geolocation' AS message;

INSERT INTO silver.geolocation
SELECT 
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    -- Standardize names to avoid "Sao Paulo" vs "SAO PAULO"
    LOWER(TRIM(TRANSLATE(
        geolocation_city, 
        'ÁÀÂÃÉÈÊÍÌÎÓÒÔÕÚÙÛÇáàâãéèêíìîóòôõúùûç', 
        'AAAAEEEIIIOOOOUUUCaaaaeeeiiioooouuuc'
    ))) AS geolocation_city,
    UPPER(TRIM(TRANSLATE(
        geolocation_state, 
        'ÁÀÂÃÉÈÊÍÌÎÓÒÔÕÚÙÛÇáàâãéèêíìîóòôõúùûç', 
        'AAAAEEEIIIOOOOUUUCaaaaeeeiiioooouuuc'
    ))) AS geolocation_state
FROM bronze.geolocation
WHERE     
    geolocation_zip_code_prefix IS NOT NULL
    AND geolocation_city IS NOT NULL;


SELECT '>> Truncating Table: Silver.order_items' AS message;

TRUNCATE TABLE silver.order_items;

SELECT '>> Inserting Data Into: Silver.order_items' AS message;

INSERT INTO silver.order_items
SELECT 
    order_id,
    CAST(order_item_id AS INT) AS order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value,
    (price + freight_value) AS total_item_value, -- Total Item(s) Cost
    CASE 
        WHEN (price + freight_value) > 0 
        THEN ROUND(CAST(freight_value AS DOUBLE) / (price + freight_value), 4)
        ELSE 0 
    END AS shipping_ratio  --Shipping Ratio (Efficiency Metric), round to 4 decimal places
FROM bronze.order_items
WHERE 
    -- Strict Financial Gates
    price >= 0 
    AND freight_value >= 0
    -- Structural Gate
    AND order_id IS NOT NULL;

SELECT '>> Truncating Table: Silver.order_payments' AS message;

TRUNCATE TABLE silver.order_payments;

SELECT '>> Inserting Data Into: Silver.order_payments' AS message;

INSERT INTO silver.order_payments 
SELECT 
    order_id,
    CAST(payment_sequential AS INT) AS payment_sequential,
    -- Clean up: 'credit_card' -> 'credit card'
    LOWER(REPLACE(payment_type, '_', ' ')) AS payment_type,
    -- Fix the '0' installments issue: Minimum should be 1
    CASE 
        WHEN CAST(payment_installments AS INT) < 1 THEN 1 
        ELSE CAST(payment_installments AS INT) 
    END AS payment_installments,
    CAST(payment_value AS DOUBLE) AS payment_value,
    CASE 
        WHEN payment_installments > 0 THEN ROUND(payment_value / payment_installments, 2)
        ELSE payment_value 
    END AS installment_value
FROM bronze.order_payments
WHERE 
    payment_value >= 0 
    AND order_id IS NOT NULL;

SELECT '>> Truncating Table: Silver.order_reviews' AS message;

TRUNCATE TABLE silver.order_reviews;

SELECT '>> Inserting Data Into: Silver.order_reviews' AS message;

INSERT INTO silver.order_reviews
WITH deduplicated_data AS (
    SELECT 
        review_id,
        order_id,
        -- 1. Standardize the score
        CAST(review_score AS INT) AS review_score,
        -- 2. Clean whitespace/newlines but preserve Portuguese letters
        TRIM(REPLACE(REPLACE(review_comment_title, '\n', ' '), '\r', ' ')) AS review_comment_title,
        TRIM(REPLACE(REPLACE(review_comment_message, '\n', ' '), '\r', ' ')) AS review_comment_message,
        review_creation_date,
        review_answer_timestamp,
        -- 3. Version Control: Identify the latest review per (Review ID + Order ID)
        ROW_NUMBER() OVER (
            PARTITION BY review_id, order_id 
            ORDER BY review_answer_timestamp DESC
        ) AS rn
    FROM bronze.order_reviews
    WHERE 
        -- Structural Integrity
        review_id IS NOT NULL 
        AND order_id IS NOT NULL
        -- 4. Your Financial/Rating Gate
        AND CAST(review_score AS INT) BETWEEN 1 AND 5
)
SELECT 
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
FROM deduplicated_data
WHERE rn = 1;

SELECT '>> Truncating Table: Silver.products' AS message;

TRUNCATE TABLE silver.products;

SELECT '>> Inserting Data Into: Silver.products' AS message;
INSERT INTO silver.products
SELECT 
    product_id,
    -- Handle missing categories with a default label
    COALESCE(TRIM(LOWER(product_category_name)), 'n/a') AS product_category_name,
    
    -- Metadata lengths (cast to INT for storage efficiency)
    CAST(product_name_lenght AS INT) AS product_name_length,
    CAST(product_description_lenght AS INT) AS product_description_length,
    CAST(product_photos_qty AS INT) AS product_photos_qty,
    
    -- Physical Dimensions
    CASE 
        WHEN product_weight_g IS NULL OR product_weight_g = 0 THEN 
            ROUND(AVG(CASE WHEN product_weight_g > 0 THEN CAST(product_weight_g AS INT) END) 
                  OVER(PARTITION BY product_category_name), 2)
        ELSE CAST(product_weight_g AS INT)
    END AS product_weight_g,
    CAST(product_length_cm AS INT) AS product_length_cm,
    CAST(product_height_cm AS INT) AS product_height_cm,
    CAST(product_width_cm AS INT) AS product_width_cm
FROM bronze.products
WHERE product_id IS NOT NULL;


SELECT '>> Truncating Table: Silver.product_category_name_translation' AS message;

TRUNCATE TABLE silver.product_category_name_translation;

SELECT '>> Inserting Data Into: Silver.product_category_name_translation' AS message;

INSERT INTO silver.product_category_name_translation
WITH deduplicated_mapping AS (
    SELECT 
        -- We give these aliases so the outer query can select them by name
        TRIM(LOWER(product_category_name)) AS product_category_name,
        TRIM(LOWER(product_category_name_english)) AS product_category_name_english,
        
        -- rn is only used here for filtering and will not be passed to the INSERT
        ROW_NUMBER() OVER (
            PARTITION BY product_category_name 
            ORDER BY product_category_name_english ASC
        ) AS rn
    FROM bronze.product_category_name_translation
    WHERE product_category_name IS NOT NULL
)
SELECT 
    product_category_name,
    product_category_name_english
FROM deduplicated_mapping
WHERE rn = 1;


SELECT '>> Truncating Table: Silver.sellers' AS message;

TRUNCATE TABLE silver.sellers;

SELECT '>> Inserting Data Into: Silver.sellers' AS message;

INSERT INTO silver.sellers
SELECT 
    seller_id,
    CAST(seller_zip_code_prefix AS INT) AS seller_zip_code_prefix,
    TRIM(LOWER(
        TRANSLATE(seller_city, 
                  'áàâãäéèêëíìîïóòôõöúùûüç', 
                  'aaaaaeeeeiiiiooooouuuuc')
    )) AS seller_city,    -- Standardizing city names: lowercase, trimmed, and removing accents
    UPPER(seller_state) AS seller_state -- Ensuring State is uppercase (e.g., 'sp' -> 'SP')
FROM bronze.sellers
WHERE seller_id IS NOT NULL;